# OpenAI SDK

source: [quickstart](https://developers.openai.com/api/docs/quickstart)

## Setup

1. Create API Key: https://platform.openai.com/api-keys
2. Insert the API key in .env
    - OPENAI_API_KEY=<...>
3. poetry add openai

## Test - OpenAI SDK

We'll use the OpenAI client:
- Is a client built on httpx.client
- Retrieves the OPENAI_API_KEY environment variable 
- Adds Idempotency key on each request (if a timeout occurs you don't pay your request twice)
- Grants http client pooling (as httpx.client)
- Retries with exponential backoffs or retry-after on selected errors (429, 5xx)
- Grants Structured output with pydantic
- Does SSE parsing when using streams (error handling is simplified with just a custom exception)

### Simple sync response

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from openai.types import Reasoning

# to add .env content to environment variables
load_dotenv()

client = OpenAI() 

response = client.responses.create(
    model="gpt-5-nano",
    input="Write a one-sente bedtime story about a unicorn",
    reasoning=Reasoning(effort="minimal"), # this avoids reasoning
    store=False # by default this is true and lets you access the previous_response_id  
)

print(f"usage: \n{response.usage.model_dump_json(indent=2)}")
print(f"\noutput text: {response.output_text}")
print(f"\n\noutput: \n{response.model_dump_json(indent=2)}")

usage: 
{
  "input_tokens": 16,
  "input_tokens_details": {
    "cache_write_tokens": 0,
    "cached_tokens": 0
  },
  "output_tokens": 70,
  "output_tokens_details": {
    "reasoning_tokens": 0
  },
  "total_tokens": 86,
  "compute_units": null
}

output text: Under the silver glow of the moon, a gentle unicorn followed a trail of twinkling fireflies to a quiet meadow where dreams grew like daisies, and as she whispered a lullaby to the night, the stars settled into a soft, sleepy hush.


output: 
{
  "id": "resp_00dfe8c0f05587e6006a96a3601f0887d2a8380792939da164",
  "created_at": 1788257120.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5-nano-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "rs_00dfe8c0f05587e6006a96a3609ab887d2a57bd6857d9d7ea4",
      "summary": [],
      "type": "reasoning",
      "content": [],
      "encrypted_content": "gAAAAABqlqNh0DS8lEcfN98-da-7emdc19nK8GizGghMdweWyNkiuyXoAo5g5H

### Simple async responses

We will run five LLM calls concurrently using the AsyncOpenAI client to receive results in the time of the slowest call.

We open just one client and reuse it to use http connection pooling.

In [ ]:
import asyncio
from openai import AsyncOpenAI
from dotenv import load_dotenv
from openai.types import Reasoning

async def tell_a_story(client: AsyncOpenAI, subject: str):
    response = await client.responses.create(
        model="gpt-5-nano",
        input=f"Write a one-sente bedtime story about a {subject}",
        reasoning=Reasoning(effort="minimal"),
        store=False
    )
    return response

# to add .env content to environment variables
load_dotenv()

subjects = ["unicorn", "pony", "candy", "llm", "me"]
responses = []

async with AsyncOpenAI() as client:
    responses = await asyncio.gather(
        *(tell_a_story(client, s) for s in subjects),
        return_exceptions=True
    )

for s,r in zip(subjects, responses):
    if isinstance(r, Exception):
        print(f"Returned error: {s} -> {r}")
    else:
        print(f"Result: {s} -> {r.output_text}")

tot_input = sum(r.usage.input_tokens for r in responses if not isinstance(r,Exception))
tot_output = sum(r.usage.output_tokens for r in responses if not isinstance(r,Exception))
print(f"usage input_tokens: {tot_input}")
print(f"usage output_tokens: {tot_output}")

Result: unicorn -> A gentle unicorn trotted under a silvery moonlit sky, brushed by whispering stars, and as she hummed a lullaby to the sleepy meadow, all the dreamless flowers drifted into slumber, promising tomorrow’s adventures.
Result: pony -> In the quiet glow of the moonlit meadow, a gentle pony named Rose bridged the dream between waking and sleeping by trotting softly beneath the stars, leaving a trail of twinkling fireflies that whispered, goodnight, little dreamer.
Result: candy -> In the hush of the twilit kitchen, a little candy sighed softly, closed its wrapper like a gentle kiss, and drifted off to dream of sunny bakeries and brave sprinkles ready to brighten tomorrow.
Result: llm -> In the quiet glow of the server room, a gentle LLM whispered a lullaby of words until the midnight code drifted into dreamlike syntax and the night hummed softly to rest.
Result: me -> Once upon a time, a gentle moonbeam found a sleepy me tucked under a cozy blanket and whispered, “Close you

## Test - GROQ with OpenAI SDK

Groq is a handy alternative to use as a test because it offers free API access with no credit card, capped by requests per minute/day and tokens per minute/day (see https://console.groq.com/docs/rate-limits).

We will use the same client as before because Groq exposes a OpenAI compatible API.

Note:
- Groq API is compatible with OpenAI Responses API (tested with groq/compound-mini model)
- Chinese providers are usually compatible with ChatCompletions API and not Responses API so you should use `client.chat.completions.create` there
- Groq refuses the OpenAI-style `reasoning` parameter returning an error if you try to pass it 

In [10]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# to add .env content to environment variables
load_dotenv()

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],         # we just have to specify the base_url and api_key to use
    base_url="https://api.groq.com/openai/v1",
) 

client.chat.completions.create

response = client.responses.create(
    model="qwen/qwen3.8-27b", # groq/compound-mini openai/gpt-oss-120b meta-llama/llama-prompt-guard-2-86m
    input="Write a one-sente bedtime story about a unicorn",
    # reasoning=Reasoning(effort="minimal"), # THIS ISN'T SUPPORTED BY GROQ
    store=False
)

print(f"usage: \n{response.usage.model_dump_json(indent=2)}")
print(f"\noutput text: {response.output_text}")
print(f"\n\noutput: \n{response.model_dump_json(indent=2)}")

usage: 
{
  "input_tokens": 22,
  "input_tokens_details": {
    "cache_write_tokens": null,
    "cached_tokens": 0
  },
  "output_tokens": 35,
  "output_tokens_details": {
    "reasoning_tokens": 0
  },
  "total_tokens": 57,
  "compute_units": null
}

output text: The gentle unicorn curled up in the moonbeam-lit meadow, her silver horn glowing softly as she drifted into a dream of starry skies and endless rainbows.


output: 
{
  "id": "resp_01m1eb7pa3e9zsezmeshc9hahs",
  "created_at": 1788261685.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "qwen/qwen3.8-27b",
  "object": "response",
  "output": [
    {
      "id": "resp_01m1eb7pa3ea08c54w224e0vay",
      "summary": [],
      "type": "reasoning",
      "content": null,
      "encrypted_content": null,
      "status": "completed"
    },
    {
      "id": "msg_01m1eb7pa3ea0s5khdk1635dhr",
      "content": [
        {
          "annotations": [],
          "text": "The gentle unico